# Step 2: build the training cache from the full raw dataset (CPU, ~20-60 min per cohort, run x3)
Why: the v1 cache stored uint8 clipped at the volume's 99.5th percentile, which saturates much of the tumour,
and it fixes one normalisation. v2 stores the raw values (float16) of the tumour slices, the full 256 plane and
whole-volume statistics. Every normalisation can then be chosen at training time without a rebuild. Only
pCR-labeled patients (1,452 of 2,070) are written by default. The other 618, mostly Duke, have no pCR outcome,
so they can be neither trained on nor scored; `ALL_PATIENTS = True` includes them anyway.

1. Session options: Accelerator **None**, Internet **ON**, secret `GITHUB_TOKEN` (or attach the code Dataset).
2. **+ Add Input** -> the raw Dataset of the cohort from step 1 (`mnj-raw-ispy1`, `mnj-raw-ispy2` or `mnj-raw-duke`;
   `breastdcedl-full` if you uploaded everything as one Dataset).
3. Set `COHORT`, then **Save Version -> Save & Run All (Commit)**. Run it three times (ispy1, ispy2, duke).
4. When it finishes: **New Dataset** from the output, named **`mnj-cache2-<cohort>`** (Private).
   Measured size per cohort (labeled / all): I-SPY1 1.1 / 1.1 GB, I-SPY2 8.2 / 8.2 GB, Duke 2.1 / 5.2 GB.

In [ ]:
# ==== PARAMETERS (edit these) ====
REPO = "AkulaReddy/MRI_chemo_response"
GIT_TAG = "vaib"           # branch (or tag) to clone
PERSON = ""               # optional: "A", "B" or "C" -> results/<PERSON>/ (empty -> results/vaib/)
COHORT = "ispy1"        # "ispy1", "ispy2" or "duke"
LIMIT = 0                # 0 = every patient (full cohort); 10 only for a quick check
ALL_PATIENTS = False     # True: also the patients without a pCR label (Duke 916 instead of 298)
RAW = ""                 # leave "" to find the BreastDCEDL_<COHORT>_min_crop input automatically

In [ ]:
# ==== SETUP (do not edit): get the repo code, install requirements ====
import base64, json, os, shutil, subprocess, sys
if PERSON not in ("A", "B", "C"):           # only names the results folder
    PERSON = "vaib"
    print('PERSON not set: results go to results/vaib/ (set PERSON = "A", "B" or "C" to change)')
_tok = None
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass

def git(*args, cwd="/tmp/repo"):
    """git with the token as a one-off header (never stored in .git/config or the output)."""
    if not _tok:
        raise RuntimeError("no GITHUB_TOKEN secret: git push/pull is not available")
    hdr = "Authorization: Basic " + base64.b64encode(f"x-access-token:{_tok}".encode()).decode()
    r = subprocess.run(["git", "-c", f"http.extraHeader={hdr}", *args], cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(_tok, "***")
    if r.returncode:
        raise RuntimeError(f"git {args[0]} failed:\n{out}")
    return out

shutil.rmtree("/tmp/repo", ignore_errors=True)
if _tok:                                    # 1) clone the branch from GitHub
    git("clone", "--quiet", "--depth", "1", "--branch", GIT_TAG, f"https://github.com/{REPO}.git", "/tmp/repo", cwd="/tmp")
    print("code: GitHub", REPO, GIT_TAG, git("log", "-1", "--format=%h %s"))
else:                                       # 2) or a code Dataset (zip of the repo) attached as input
    import glob
    hits = [h for d in range(1, 6) for h in glob.glob("/kaggle/input/" + "*/" * d + "src/vitonly/__init__.py")]
    assert hits, ("no GITHUB_TOKEN secret and no code Dataset attached. Either add the secret (Add-ons -> Secrets) "
                  "or attach the Dataset made from mnj-code-vaib.zip (see kaggle/vaib/README.md)")
    code_root = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
    shutil.copytree(code_root, "/tmp/repo")
    print("code: attached Dataset", code_root)
os.chdir("/tmp/repo")
sys.path.insert(0, "/tmp/repo")         # so `from src...` works in this kernel
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
if not RAW:
    want = f"BreastDCEDL_{COHORT.upper()}_min_crop"
    hits = []
    for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
        if root.count("/") > 7:
            dirs[:] = []; continue
        hits += [os.path.join(root, n) for n in dirs + files if n.startswith(want)]
        dirs[:] = [d for d in dirs if not d.startswith("BreastDCEDL_")]
    assert hits, f"no {want} under /kaggle/input: attach mnj-raw-{COHORT} or set RAW"
    RAW = hits[0] if os.path.isdir(hits[0]) else os.path.dirname(hits[0])
OUT = f"/kaggle/working/cache2_{COHORT}"
cmd = [sys.executable, "scripts/build_cache_v2.py", "--cohort", COHORT, "--raw", RAW, "--out", OUT, "--workers", "4"]
if LIMIT: cmd += ["--limit", str(LIMIT)]
if ALL_PATIENTS: cmd.append("--all-patients")
print(" ".join(cmd)); subprocess.run(cmd, check=True)

In [ ]:
import pandas as pd
idx = pd.read_csv(f"{OUT}/cache_index.csv")
print(json.dumps(json.load(open(f"{OUT}/cache_meta.json")), indent=1))
ok = idx[idx.status == "ok"]
print("tumour-box share clipped by the v1 scaling (per phase):")
display(ok[["sat_pre", "sat_early", "sat_late"]].describe().round(3))
display(idx[idx.status == "error"].head(20))